In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib as mpl
import geopandas as gpd
import rasterio
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cartopy.feature as cfeature
import pandas as pd

mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['font.size'] = 12

mask_path = r'PATH_TO_MASK_FILE'
shapefile_path = r'PATH_TO_SHAPEFILE'

boundaries = [-1, -0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
colors = [
    "#2D88A9", "#45C5F7", "#7ED1EF", "#B3E2FB", "#DAF1FD",
    "#FCEEC1", "#FBE7A5", "#FAD07A", "#F3A800", "#9E6E12"
]
cmap = ListedColormap(colors, 'indexed')
norm = BoundaryNorm(boundaries, ncolors=len(colors), clip=True)

percentage_data = []

def max_balance_method(percentages, total=100, precision=2):
    percentages_decimal = [p / total for p in percentages]
    total_decimal = sum(percentages_decimal)
    deficit = 1 - total_decimal
    adjustments = [p * deficit / total_decimal for p in percentages_decimal]
    adjusted_percentages = [
        round(p * total + a * total, precision)
        for p, a in zip(percentages_decimal, adjustments)
    ]
    total_adjusted = sum(adjusted_percentages)
    diff = total - total_adjusted
    if diff != 0:
        adjusted_percentages[-1] += diff
    adjusted_percentages = [max(0, val) for val in adjusted_percentages]
    return adjusted_percentages

for start_year in range(2001, 2012):
    end_year = start_year + 9
    nc_file_name = f'max_corr_{start_year}-{end_year}_Spring.nc'
    nc_file_path = f'PATH_TO_NETCDF_DIR/{nc_file_name}'

    ds = xr.open_dataset(nc_file_path)

    max_correlation = ds['max_correlation'].values
    lons = ds['lon'].values
    lats = ds['lat'].values

    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        mask = np.flipud(mask)

    max_correlation[(mask == 0) | (max_correlation == 0)] = np.nan

    lat_mean_corr = np.nanmean(max_correlation, axis=1)
    lon_mean_corr = np.nanmean(max_correlation, axis=0)

    fig = plt.figure(figsize=(14, 10), dpi=600)
    proj = ccrs.PlateCarree()
    ax = fig.add_subplot(2, 1, 1, projection=proj)

    ax.pcolormesh(
        lons, lats, max_correlation,
        shading='auto',
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree()
    )
    ax.set_title(f'{start_year}-{end_year} MAM', fontsize=60)
    ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='whitesmoke', edgecolor='none', zorder=0)
    ax.contourf(
        lons, lats, max_correlation,
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        zorder=1
    )
    ax.add_feature(cfeature.LAND, facecolor='none', edgecolor='black', zorder=4)

    lon_ticks = np.linspace(-180, 180, 7)
    lat_ticks = np.linspace(-90, 90, 7)

    ax.set_xticks(lon_ticks, crs=proj)
    ax.set_yticks(lat_ticks, crs=proj)

    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.yaxis.set_ticks_position('right')
    ax.yaxis.set_label_position('right')
    ax.tick_params(axis='x', direction='in')
    ax.tick_params(axis='y', direction='in')

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = True
    gl.bottom_labels = True
    gl.left_labels = False
    gl.xlabel_style = {'rotation': 0, 'ha': 'center'}
    gl.ylabel_style = {'rotation': 270, 'va': 'center'}

    latitudes = ds['lat'].values
    lat_weights = np.cos(np.deg2rad(latitudes))
    lat_weights = np.tile(lat_weights[:, np.newaxis], (1, len(lons)))

    flat_correlation = max_correlation.flatten()
    flat_weights = lat_weights.flatten()

    n_intervals = len(boundaries) - 1
    percentages = np.zeros(n_intervals)

    for i in range(n_intervals):
        lower = boundaries[i]
        upper = boundaries[i + 1]
        mask_interval = (
            (flat_correlation >= lower) &
            (flat_correlation < upper) &
            ~np.isnan(flat_correlation)
        )
        total_weight = np.sum(flat_weights[mask_interval])
        percentages[i] = total_weight

    total_weight = np.sum(flat_weights[~np.isnan(flat_correlation)])
    percentages = (percentages / total_weight) * 100
    percentages = np.round(percentages, 2)
    percentages = max_balance_method(percentages.tolist(), total=100, precision=2)

    labels = [f'{boundaries[i]:.1f}-{boundaries[i+1]:.1f}' for i in range(n_intervals)]

    ax_hist = fig.add_axes([0.25, 0.53, 0.1, 0.17])
    bars = ax_hist.barh(labels, percentages, color=[colors[i] for i in range(n_intervals)])

    for i, (value, bar) in enumerate(zip(percentages, bars)):
        bar_width = bar.get_width()
        if bar_width >= 15:
            x_pos = bar_width
            ha = 'right'
        else:
            x_pos = bar_width + 1
            ha = 'left'
        ax_hist.text(x_pos, i, f'{value:.2f}%', va='center', ha=ha)

    ax_hist.set_title('Percentage Distribution', fontsize=8)
    ax_hist.set_xlabel('Percentage', fontsize=8)
    ax_hist.set_ylabel('Correlation Range', fontsize=8)
    ax_hist.tick_params(axis='both', which='major', labelsize=8)
    ax_hist.patch.set_alpha(0.5)
    ax_hist.tick_params(axis='y', labelrotation=45)

    ax2 = fig.add_axes([0.825, 0.482, 0.05, 0.424])
    ax2.barh(
        lats,
        lat_mean_corr,
        height=0.5,
        color=np.where(lat_mean_corr > 0, '#FAD07A', '#45C5F7')
    )

    ax2.set_yticks(np.linspace(-90, 90, 7))
    ax2.set_yticklabels([])
    ax2.set_ylim(-90, 90)
    ax2.set_xlim(-1, 1)
    ax2.set_xticks(np.linspace(-1, 1, 3))
    ax2.set_xlabel('', fontname='Times New Roman')
    ax2.set_title('', rotation=270)
    ax2.tick_params(axis='x', direction='in')
    ax2.tick_params(axis='y', direction='in', pad=9)

    plt.tight_layout()

    output_filename = f'max_correlation_{start_year}-{end_year}_Spring.jpg'
    output_path = f'PATH_TO_OUTPUT_DIR/{output_filename}'
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1, dpi=600)
    plt.show()

    percentage_data.append(percentages[::-1])

    output_csv_path = r'PATH_TO_OUTPUT_CSV/percentage_distribution.csv'
    header = ','.join([
        '0.8-1.0', '0.6-0.8', '0.4-0.6', '0.2-0.4', '0.0-0.2',
        '-0.2-0.0', '-0.4--0.2', '-0.6--0.4', '-0.8--0.6', '-1.0--0.8'
    ])
    np.savetxt(
        output_csv_path,
        percentage_data,
        delimiter=',',
        header=header,
        comments='',
        fmt='%.2f'
    )


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib as mpl
import rasterio
import cartopy.feature as cfeature
import pandas as pd

mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['font.size'] = 12

mask_path = r'PATH_TO_MASK_FILE'
shapefile_path = r'PATH_TO_SHAPEFILE'

boundaries = [1, 2, 3, 6, 12, 18, 24, 48]
colors = ["#8BD1F0", "#C5DBED", "#92CE4F", "#D5ECBB", "#FFEFD0", "#FEDD9E", "#FDAA11"]
cmap = ListedColormap(colors, 'indexed')
norm = BoundaryNorm(boundaries, ncolors=len(colors), clip=True)

percentage_data = []

def max_balance_method(percentages, total=100, precision=2):
    percentages_decimal = [p / total for p in percentages]
    total_decimal = sum(percentages_decimal)
    deficit = 1 - total_decimal
    adjustments = [p * deficit / total_decimal for p in percentages_decimal]
    adjusted_percentages = [
        round(p * total + a * total, precision)
        for p, a in zip(percentages_decimal, adjustments)
    ]
    total_adjusted = sum(adjusted_percentages)
    diff = total - total_adjusted
    if diff != 0:
        adjusted_percentages[-1] += diff
    adjusted_percentages = [max(0, val) for val in adjusted_percentages]
    return adjusted_percentages

for start_year in range(2001, 2012):
    end_year = start_year + 9

    nc_file_name = f'max_corr_{start_year}-{end_year}_Spring.nc'
    nc_file_path = f'PATH_TO_NETCDF_DIR/{nc_file_name}'

    ds = xr.open_dataset(nc_file_path)

    time_scale = ds['time_scale'].values
    lons = ds['lon'].values
    lats = ds['lat'].values

    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        mask = np.flipud(mask)

    time_scale[(mask == 0) | (time_scale == 0)] = np.nan

    lat_mean_time = np.nanmean(time_scale, axis=1)
    lon_mean_time = np.nanmean(time_scale, axis=0)

    fig = plt.figure(figsize=(14, 10), dpi=600)
    proj = ccrs.PlateCarree()
    ax = fig.add_subplot(2, 1, 1, projection=proj)

    ax.pcolormesh(
        lons, lats, time_scale,
        shading='auto',
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree()
    )
    ax.set_title(f'{start_year}-{end_year} MAM', fontsize=60)
    ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='whitesmoke', edgecolor='none', zorder=0)
    ax.pcolormesh(
        lons, lats, time_scale,
        shading='auto',
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        zorder=1
    )
    ax.add_feature(cfeature.LAND, facecolor='none', edgecolor='black', zorder=4)

    lon_ticks = np.linspace(-180, 180, 7)
    lat_ticks = np.linspace(-90, 90, 7)
    ax.set_xticks(lon_ticks, crs=proj)
    ax.set_yticks(lat_ticks, crs=proj)

    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.yaxis.set_ticks_position('right')
    ax.yaxis.set_label_position('right')
    ax.tick_params(axis='x', direction='in')
    ax.tick_params(axis='y', direction='in')

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = True
    gl.bottom_labels = True
    gl.left_labels = False
    gl.xlabel_style = {'rotation': 0, 'ha': 'center'}
    gl.ylabel_style = {'rotation': 270, 'va': 'center'}

    colors = ["#8BD1F0", "#C5DBED", "#92CE4F", "#D5ECBB", "#FFEFD0", "#FEDD9E", "#FDAA11"]
    boundaries = [1, 2, 3, 6, 12, 18, 24, 48]

    latitudes = ds['lat'].values
    lat_weights = np.cos(np.deg2rad(latitudes))
    lat_weights = np.tile(lat_weights[:, np.newaxis], (1, len(lons)))

    flat_time_scale = time_scale.flatten()
    flat_weights = lat_weights.flatten()

    n_intervals = len(boundaries) - 1
    percentages = np.zeros(n_intervals)

    for i in range(n_intervals):
        lower = boundaries[i]
        upper = boundaries[i + 1]
        mask_interval = (
            (flat_time_scale >= lower) &
            (flat_time_scale < upper) &
            ~np.isnan(flat_time_scale)
        )
        total_weight = np.sum(flat_weights[mask_interval])
        percentages[i] = total_weight

    total_weight = np.sum(flat_weights[~np.isnan(flat_time_scale)])
    percentages = (percentages / total_weight) * 100
    percentages = np.round(percentages, 2)
    percentages = max_balance_method(percentages.tolist(), total=100, precision=2)

    labels = ['1', '2-3', '3-6', '6-12', '12-18', '18-24', '24-48']

    ax_hist = fig.add_axes([0.25, 0.53, 0.1, 0.17])
    bars = ax_hist.barh(labels, percentages, color=[colors[i] for i in range(n_intervals)])

    for i, (value, bar) in enumerate(zip(percentages, bars)):
        bar_width = bar.get_width()
        if bar_width >= 15:
            x_pos = bar_width
            ha = 'right'
        else:
            x_pos = bar_width + 1
            ha = 'left'
        ax_hist.text(x_pos, i, f'{value:.2f}%', va='center', ha=ha)

    ax_hist.set_title('Percentage Distribution', fontsize=8)
    ax_hist.set_xlabel('Percentage', fontsize=8)
    ax_hist.set_ylabel('Time Scale Range', fontsize=8)
    ax_hist.tick_params(axis='both', which='major', labelsize=8)
    ax_hist.patch.set_alpha(0.5)
    ax_hist.tick_params(axis='y', labelrotation=45)

    plt.tight_layout()

    output_filename = f'time_scale_{start_year}-{end_year}_Spring.jpg'
    output_path = f'PATH_TO_OUTPUT_DIR/{output_filename}'
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1, dpi=600)
    plt.show()

    percentage_data.append(percentages[::-1])

    output_csv_path = r'PATH_TO_OUTPUT_CSV/percentage_distribution_timescale_Spring.csv'
    header = ','.join(['1', '2-3', '3-6', '6-12', '12-18', '18-24', '24-48'])
    np.savetxt(
        output_csv_path,
        percentage_data,
        delimiter=',',
        header=header,
        comments='',
        fmt='%.2f'
    )
